<a href="https://colab.research.google.com/github/Bernpro/Bernie/blob/main/Accounting_Migration_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# 1. GENERATE MOCK DATA FOR THE MIGRATION TEST (TEST 1)
mock_buildium = pd.DataFrame({
    'Address Line 1': ['123 Main St', '456 Oak Ave', '789 Pine Rd'],
    'City': ['Phoenix', 'Tucson', 'Mesa'],
    'State': ['AZ', 'AZ', 'AZ'],
    'Zip Code': [85001, 85701, 45],  # intentionally messed up zip format '45' to test cleaning logic
    'Rental Owner': ['John Doe', 'Jane Smith', 'Bob Johnson']
})
mock_buildium.to_csv('buildium_properties_export.csv', index=False)

# 2. GENERATE MOCK DATA FOR THE LEDGER AUDIT TEST (TEST 2)
mock_bank = pd.DataFrame({
    'Transaction_ID': ['TXN101', 'TXN102', 'TXN103'],
    'Date': ['2026-05-01', '2026-05-02', '2026-05-03'],
    'Description': ['Rent Payment Apt 1A', 'Rent Payment Apt 2B', 'ATM Cash Deposit'],
    'Amount': [1200, 1450, 500]
})
mock_bank.to_excel('bank_statement.xlsx', index=False)

mock_ledger = pd.DataFrame({
    'Reference_ID': ['TXN101', 'TXN102', 'TXN104'], # TXN104 exists here but skipped the bank!
    'Amount': [1200, 1450, 450]
})
mock_ledger.to_csv('software_ledger.csv', index=False)

print("🎉 Success! All test files have been generated in your Colab environment.")




# Load the fake file we generated in the setup step
buildium_df = pd.read_csv('buildium_properties_export.csv')

# Clean up data formatting (zfill adds leading zeros if a zip code is truncated)
buildium_df['Zip Code'] = buildium_df['Zip Code'].astype(str).str.zfill(5)

# Build the structural mapping layout required by Rentvine
rentvine_df = pd.DataFrame()
rentvine_df['Property_Street'] = buildium_df['Address Line 1']
rentvine_df['Property_City'] = buildium_df['City']
rentvine_df['Property_State'] = buildium_df['State']
rentvine_df['Property_Zip'] = buildium_df['Zip Code']

# Split names cleanly by spaces
rentvine_df['Owner_First_Name'] = buildium_df['Rental Owner'].str.split(' ').str[0]
rentvine_df['Owner_Last_Name'] = buildium_df['Rental Owner'].str.split(' ').str[1]

# Save the polished output
rentvine_df.to_csv('ready_for_rentvine_upload.csv', index=False)

print("--- TRANSFORMED RENTVINE DATA VIEW ---")
print(rentvine_df.head())




# Load our mock financial logs
bank_df = pd.read_excel('bank_statement.xlsx')
ledger_df = pd.read_csv('software_ledger.csv')

# Find a ledger transaction that was recorded but never actually settled at the bank
missing_from_bank = ledger_df[~ledger_df['Reference_ID'].isin(bank_df['Transaction_ID'])]

# Find an influx of money at the bank that the property manager forgot to document in the software
unrecorded_cash = bank_df[~bank_df['Transaction_ID'].isin(ledger_df['Reference_ID'])]

print("⚠️ MISSED BY THE BANK (Check for bounced checks/un-cleared items):")
print(missing_from_bank)
print("\n⚠️ UNRECORDED IN SOFTWARE (Cash is in the bank, but where is it on the ledger?):")
print(unrecorded_cash[['Date', 'Description', 'Amount']])


def calculate_disposition(tenant_name, deposit, unpaid_rent, damages):
    total_deductions = unpaid_rent + damages
    refund = deposit - total_deductions

    print(f"=== Disposition Report for {tenant_name} ===")
    print(f"Initial Deposit Held: ${deposit:,.2f}")
    print(f"Total Deductions:     ${total_deductions:,.2f} (Rent: ${unpaid_rent}, Damages: ${damages})")

    if refund >= 0:
        return f"👉 Action Required: Refund Tenant ${refund:,.2f}"
    else:
        return f"👉 Action Required: Charge Tenant Portfolio ${abs(refund):,.2f}"

# Test Case A: Tenant leaves the apartment clean but missed their final rent payment
print(calculate_disposition("Apartment 3C - Alex Medina", deposit=1500, unpaid_rent=1200, damages=0))
print("-" * 50)

# Test Case B: Tenant paid all rent but damaged the drywall
print(calculate_disposition("Apartment 12A - Sarah Jenkins", deposit=1000, unpaid_rent=0, damages=1450))




🎉 Success! All test files have been generated in your Colab environment.
--- TRANSFORMED RENTVINE DATA VIEW ---
  Property_Street Property_City Property_State Property_Zip Owner_First_Name  \
0     123 Main St       Phoenix             AZ        85001             John   
1     456 Oak Ave        Tucson             AZ        85701             Jane   
2     789 Pine Rd          Mesa             AZ        00045              Bob   

  Owner_Last_Name  
0             Doe  
1           Smith  
2         Johnson  
⚠️ MISSED BY THE BANK (Check for bounced checks/un-cleared items):
  Reference_ID  Amount
2       TXN104     450

⚠️ UNRECORDED IN SOFTWARE (Cash is in the bank, but where is it on the ledger?):
         Date       Description  Amount
2  2026-05-03  ATM Cash Deposit     500
=== Disposition Report for Apartment 3C - Alex Medina ===
Initial Deposit Held: $1,500.00
Total Deductions:     $1,200.00 (Rent: $1200, Damages: $0)
👉 Action Required: Refund Tenant $300.00
----------------------